In [ ]:
import os, random
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
import matplotlib.pyplot as plt

SEED = 42
def set_seed(seed=SEED):
    random.seed(seed); np.random.seed(seed)
    torch.manual_seed(seed); torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark     = False

set_seed()
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device: {device}  |  PyTorch: {torch.__version__}')

In [ ]:
# ╔══════════════════════════════════════════════════════════════════════════╗
# ║  GWGRU-Net  —  root-cause-fixed version                                 ║
# ╠══════════════════════════════════════════════════════════════════════════╣
# ║  DIAGNOSIS OF PREVIOUS FAILURES:                                         ║
# ║  • v25 (WaveBlock+STFormer):  BatchNorm2d → NaN at ep35. Best 15.50     ║
# ║  • PASTA-Net v1: d_skip=256, n_layers=12, ks=2, gcn_order=2             ║
# ║    → Every hyperparameter 2-4× SMALLER than v20. Result: 15.45          ║
# ║    Global attention (SpatialAttn+TemporalAttn) → overfitting              ║
# ║    Cosine LR → reduces LR even when val still improving                  ║
# ║                                                                           ║
# ║  FIXES:                                                                   ║
# ║  1. d_skip=512, n_layers=16, ks=4, gcn_order=4 — restore v20 capacity   ║
# ║  2. GroupNorm (no NaN ever, proven in PASTA)                             ║
# ║  3. GRU temporal decoder: reads all 12 input steps → better long-horizon ║
# ║  4. NO global attention (proved to overfit)                               ║
# ║  5. Learnable skip weights (let model emphasise useful dilations)         ║
# ║  6. ReduceLROnPlateau+warmup (proven stable, from v24)                   ║
# ║  7. Combined GRU + last-skip output (local + global temporal features)   ║
# ╚══════════════════════════════════════════════════════════════════════════╝

class Config:
    # ── Paths ──────────────────────────────────────────────────────────────
    data_path    = "/home/user/Downloads/PEMS08.npz"
    adj_csv_path = "/home/user/Downloads/PEMS08.csv"
    best_path    = 'gwgru_v1_best.pt'

    # ── Dataset ─────────────────────────────────────────────────────────────
    num_nodes   = 170
    in_features = 3       # flow + speed + occupancy
    seq_len     = 12      # 60-min input (standard benchmark)
    pred_len    = 12      # 60-min prediction
    feature_idx = 0       # predict flow only
    train_ratio = 0.7
    val_ratio   = 0.1
    HOUR_OFFSET = 12      # 1 hour = 12 × 5-min slots
    DAY_OFFSET  = 288     # 1 day  = 288 × 5-min slots

    # ── Model — v20 hyperparameters (proven, NOT downscaled) ───────────────
    d_model    = 96       # hidden dim (96/16=6 → GroupNorm safe)
    d_skip     = 512      # skip dim — KEY: was 256 in PASTA (too small!)
    d_gru      = 256      # GRU hidden for temporal decoder
    n_layers   = 16       # WaveBlocks — KEY: was 12 in PASTA (too few!)
    kernel_size = 4       # TCN kernel — KEY: was 2 in PASTA (too small!)
    adp_emb    = 20       # adaptive graph embed — KEY: was 16 in PASTA
    gcn_order  = 4        # GCN diffusion hops — KEY: was 2 in PASTA!
    n_supports = 3        # A_fwd, A_bwd, A_adp
    dropout    = 0.15
    d_time     = 32       # tod + dow embedding dim

    # ── Training ───────────────────────────────────────────────────────────
    batch_size   = 32
    lr           = 7e-4
    warmup_eps   = 5      # linear warmup (avoids early spikes)
    epochs       = 300
    patience     = 60     # generous patience for plateau schedule
    weight_decay = 1e-4
    huber_delta  = 1.0
    grad_clip    = 3.0
    # ReduceLROnPlateau (NOT cosine — cosine reduces LR even when improving)
    plat_patience = 20
    plat_factor   = 0.7
    min_lr        = 1e-5

cfg    = Config()
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'GWGRU-Net  |  d={cfg.d_model} | d_skip={cfg.d_skip} | n_layers={cfg.n_layers}')
print(f'kernel={cfg.kernel_size} | gcn_order={cfg.gcn_order} | adp_emb={cfg.adp_emb}')
print(f'Baseline -> MAE=13.114 | RMSE=22.623 | MAPE=8.471%')

In [ ]:
# ═══════════════════════════════════════════════════════════════════════════
# DATA  — 3-stream dataset (proven working, same as all previous versions)
# ═══════════════════════════════════════════════════════════════════════════

def load_pems08(cfg):
    import pandas as pd
    raw  = np.load(cfg.data_path)
    data = raw['data'].astype(np.float32)
    T, N, F = data.shape
    print(f'Raw shape: {data.shape}')
    mean_np = data.mean(axis=0)          # (N, F)
    std_np  = data.std(axis=0) + 1e-8
    data_norm = (data - mean_np[None]) / std_np[None]

    A_dist = None
    if os.path.exists(cfg.adj_csv_path):
        df    = pd.read_csv(cfg.adj_csv_path)
        A_raw = np.zeros((N, N), dtype=np.float32)
        for _, r in df.iterrows():
            i, j, c = int(r['from']), int(r['to']), float(r['cost'])
            if i < N and j < N:
                A_raw[i,j] = c; A_raw[j,i] = c
        nz    = A_raw[A_raw > 0]
        sigma = nz.std() if len(nz) > 0 else 1.0
        A_dist = np.exp(-(A_raw**2) / (sigma**2 + 1e-8))
        np.fill_diagonal(A_dist, 0.0)
        A_dist = A_dist / (A_dist.sum(1, keepdims=True) + 1e-8)
        print(f'Adj: {(A_dist>0).sum()} edges, avg_deg={(A_dist>0).sum()/N:.1f}')
    else:
        A_dist = np.eye(N, dtype=np.float32)
        print('WARNING: CSV not found — identity adj')
    return data_norm, mean_np, std_np, A_dist


class TrafficDataset3S(Dataset):
    def __init__(self, data, seq_len, pred_len, feature_idx,
                 hour_off, day_off, split_start=0, split_end=None):
        self.data, self.sl, self.pl = data, seq_len, pred_len
        self.fi, self.ho, self.do   = feature_idx, hour_off, day_off
        T = len(data); split_end = split_end or T
        eff_start = max(split_start, day_off)
        self.idx  = list(range(eff_start, split_end - seq_len - pred_len + 1))

    def __len__(self): return len(self.idx)

    def __getitem__(self, n):
        i, S = self.idx[n], self.sl
        x_rec  = self.data[i        : i+S         ].copy()
        x_hour = self.data[i-self.ho: i-self.ho+S ].copy()
        x_day  = self.data[i-self.do: i-self.do+S ].copy()
        y      = self.data[i+S : i+S+self.pl, :, self.fi].copy()
        tod    = np.array([(i+t) % 288      for t in range(S)], dtype=np.int64)
        dow    = np.array([((i+t)//288) % 7 for t in range(S)], dtype=np.int64)
        return (torch.from_numpy(x_rec),  torch.from_numpy(x_hour),
                torch.from_numpy(x_day),  torch.from_numpy(y),
                torch.from_numpy(tod),    torch.from_numpy(dow))


def build_dataloaders(cfg):
    set_seed()
    data, mean_np, std_np, A_dist = load_pems08(cfg)
    T  = len(data); t1 = int(T*cfg.train_ratio); t2 = int(T*(cfg.train_ratio+cfg.val_ratio))
    kw    = dict(batch_size=cfg.batch_size, num_workers=2, pin_memory=True)
    ds_kw = dict(data=data, seq_len=cfg.seq_len, pred_len=cfg.pred_len,
                 feature_idx=cfg.feature_idx, hour_off=cfg.HOUR_OFFSET, day_off=cfg.DAY_OFFSET)
    dl_tr = DataLoader(TrafficDataset3S(**ds_kw, split_start=0,  split_end=t1), shuffle=True,  **kw)
    dl_va = DataLoader(TrafficDataset3S(**ds_kw, split_start=t1, split_end=t2), shuffle=False, **kw)
    dl_te = DataLoader(TrafficDataset3S(**ds_kw, split_start=t2, split_end=T ), shuffle=False, **kw)
    print(f'Train={len(dl_tr.dataset)} | Val={len(dl_va.dataset)} | Test={len(dl_te.dataset)}')
    return dl_tr, dl_va, dl_te, mean_np, std_np, A_dist

print('3-stream dataset defined.')

In [ ]:
# ═══════════════════════════════════════════════════════════════════════════
# BUILDING BLOCKS — GWGRU-Net v1
# ═══════════════════════════════════════════════════════════════════════════

class DiffusionGCN(nn.Module):
    """
    K-hop bidirectional diffusion GCN.
    order=4 (restored from v20): captures 4-hop neighbourhood — richer than
    the order=2 used in PASTA which limited spatial range too much.
    """
    def __init__(self, d_in, d_out, n_supports=3, order=4, dropout=0.15):
        super().__init__()
        self.mlp   = nn.Linear(d_in * (1 + n_supports * order), d_out)
        self.drop  = nn.Dropout(dropout)
        self.order = order

    def forward(self, x, supports):   # x: (B*S, N, d)
        hs = [x]
        for A in supports:
            h = x
            for _ in range(self.order):
                h = torch.einsum('nm,bmd->bnd', A, h)
                hs.append(h)
        return self.drop(self.mlp(torch.cat(hs, dim=-1)))


class WaveBlock(nn.Module):
    """
    Gated Dilated TCN + DiffusionGCN with GroupNorm.

    WHY GroupNorm instead of BatchNorm2d:
      BatchNorm2d normalises across the batch dimension → when batch statistics
      are noisy (especially under AMP / float16), running stats diverge → NaN.
      GroupNorm normalises within each (batch, channel-group) independently,
      so it is BATCH-SIZE INDEPENDENT and NEVER produces NaN under AMP.
      We use num_groups=16 because d_model=96 → 96/16=6 channels per group.

    kernel_size=4 (restored from v20): larger kernel = richer temporal
    context per block compared to ks=2 used in PASTA.
    """
    def __init__(self, d, d_skip, ks, dilation, n_sup, order, dropout):
        super().__init__()
        pad = (ks - 1) * dilation
        self.pad_len   = pad
        self.f_conv    = nn.Conv2d(d, d, (1, ks), dilation=(1, dilation))
        self.g_conv    = nn.Conv2d(d, d, (1, ks), dilation=(1, dilation))
        self.gcn       = DiffusionGCN(d, d, n_sup, order, dropout)
        self.gn        = nn.GroupNorm(16, d)    # 96/16=6 per group, always stable
        self.skip_conv = nn.Conv2d(d, d_skip, (1, 1))
        self.res_conv  = nn.Conv2d(d, d,      (1, 1))
        self.drop      = nn.Dropout(dropout)

    def forward(self, x, supports):              # x: (B, d, N, S)
        res = x
        xp  = F.pad(x, [self.pad_len, 0])       # causal left-pad
        f   = torch.tanh   (self.f_conv(xp))
        g   = torch.sigmoid(self.g_conv(xp))
        x   = self.drop(f * g)
        B, d, N, S = x.shape
        xg  = x.permute(0, 3, 2, 1).reshape(B*S, N, d)
        xg  = self.gcn(xg, supports)
        x   = xg.reshape(B, S, N, d).permute(0, 3, 2, 1)  # (B, d, N, S)
        x   = self.gn(x)                        # GroupNorm — NEVER NaN
        return self.res_conv(x) + res, self.skip_conv(x)


print('DiffusionGCN | WaveBlock (GroupNorm, ks=4, order=4) defined.')

In [ ]:
# ═══════════════════════════════════════════════════════════════════════════
# GWGRU-Net — Graph WaveNet + GRU Temporal Decoder
#
# Architecture:
#  1. 3-stream input (rec + hour + day) → additive fusion → (B, d, N, S)
#     + node_emb + tod_emb + dow_emb
#  2. 16 WaveBlocks (dilated TCN + DiffGCN + GroupNorm) → skip list
#  3. Learnable weighted skip sum → (B, 512, N, S)
#  4a. Last timestep → h_local  (B, 512, N, 1)  [local temporal info]
#  4b. GRU over S timesteps → h_n  (B, 256, N, 1) [global temporal memory]
#  5. Concat [h_local || h_gru] → (B, 768, N, 1)
#  6. Output MLP: FC(768→512)→ReLU→FC(512→12) → (B, 12, N)
#
# Key novelties over v25/PASTA:
#  ✓ v20 capacity (d_skip=512, n=16, ks=4, order=4) — not downscaled
#  ✓ GroupNorm — zero NaN risk (no BatchNorm)
#  ✓ GRU decoder — reads full 12-step sequence, better long-horizon
#  ✓ Learnable skip weights — model chooses which dilations matter
#  ✓ Combined local + GRU output — short AND long horizon info
#  ✗ NO global attention — removed (proved to cause overfitting in PASTA)
#  ✗ NO complex cross-attention fusion — simple additive is more stable
# ═══════════════════════════════════════════════════════════════════════════

class GWGRUNet(nn.Module):
    def __init__(self, cfg, A_dist_np):
        super().__init__()
        N = cfg.num_nodes
        d = cfg.d_model

        # ── Static adjacency: forward + backward diffusion ───────────────
        A   = torch.FloatTensor(A_dist_np)
        Df  = A.sum(1, keepdim=True).clamp(min=1e-8)
        Db  = A.T.sum(1, keepdim=True).clamp(min=1e-8)
        self.register_buffer('A_fwd', A   / Df)
        self.register_buffer('A_bwd', A.T / Db)

        # ── Adaptive adjacency — learnable node embeddings ───────────────
        self.E1 = nn.Parameter(torch.randn(N, cfg.adp_emb) * 0.01)
        self.E2 = nn.Parameter(torch.randn(N, cfg.adp_emb) * 0.01)

        # ── 3-stream input projections (Conv2d on spatial-temporal grid) ─
        self.sc_rec  = nn.Conv2d(cfg.in_features, d, (1, 1))
        self.sc_hour = nn.Conv2d(cfg.in_features, d, (1, 1))
        self.sc_day  = nn.Conv2d(cfg.in_features, d, (1, 1))

        # ── Node + temporal embeddings ────────────────────────────────────
        self.node_emb = nn.Parameter(torch.randn(1, d, N, 1) * 0.01)
        self.tod_emb  = nn.Embedding(288, cfg.d_time)
        self.dow_emb  = nn.Embedding(7,   cfg.d_time)
        self.time_proj = nn.Linear(cfg.d_time * 2, d)

        # ── 16 WaveBlocks (v20 capacity: ks=4, dilations=[1,2,4,8]×4) ───
        dilations = ([1, 2, 4, 8] * 4)[:cfg.n_layers]
        self.blocks = nn.ModuleList([
            WaveBlock(d, cfg.d_skip, cfg.kernel_size, dil,
                      cfg.n_supports, cfg.gcn_order, cfg.dropout)
            for dil in dilations
        ])

        # ── Learnable skip weights (let model emphasise useful dilations) ─
        # Initialised to 0 → softmax → equal weights at start
        self.log_skip_w = nn.Parameter(torch.zeros(cfg.n_layers))

        # ── GRU temporal decoder ──────────────────────────────────────────
        # Reads the full S=12 sequence of skip features
        # → last hidden encodes full temporal history with gating
        self.gru = nn.GRU(
            input_size=cfg.d_skip,
            hidden_size=cfg.d_gru,
            num_layers=1,
            batch_first=True,
            dropout=0.0
        )

        # ── Output MLP: concat(local_skip, gru_hidden) → pred ────────────
        combo_dim = cfg.d_skip + cfg.d_gru     # 512 + 256 = 768
        self.end_conv1 = nn.Conv2d(combo_dim,    512,          (1, 1))
        self.end_conv2 = nn.Conv2d(512,           cfg.pred_len, (1, 1))

        self._init_weights()

    def _init_weights(self):
        for m in self.modules():
            if isinstance(m, (nn.Conv2d, nn.Linear)):
                nn.init.xavier_uniform_(m.weight)
                if m.bias is not None: nn.init.zeros_(m.bias)
        nn.init.normal_(self.tod_emb.weight, std=0.01)
        nn.init.normal_(self.dow_emb.weight, std=0.01)

    def get_supports(self):
        A_adp = F.softmax(F.relu(self.E1 @ self.E2.T), dim=-1)
        return [self.A_fwd, self.A_bwd, A_adp]

    def forward(self, x_rec, x_hour, x_day, tod, dow):
        # x_*: (B, S, N, F) → permute to (B, F, N, S) for Conv2d
        def to4d(x): return x.permute(0, 3, 2, 1)

        # 1. Additive 3-stream fusion (proven stable, no cross-attention)
        x  = (self.sc_rec (to4d(x_rec )) +
              self.sc_hour(to4d(x_hour)) +
              self.sc_day (to4d(x_day )))   # (B, d, N, S)
        x  = x + self.node_emb              # broadcast spatial identity

        # 2. Temporal embedding (tod + dow → d)
        te = torch.cat([self.tod_emb(tod), self.dow_emb(dow)], dim=-1)  # (B,S,2*dt)
        te = self.time_proj(te).permute(0, 2, 1).unsqueeze(2)           # (B,d,1,S)
        x  = x + te

        # 3. 16 WaveBlocks → accumulate skips
        sup   = self.get_supports()
        skips = []
        for blk in self.blocks:
            x, skip = blk(x, sup)
            skips.append(skip)              # each: (B, d_skip, N, S)

        # 4. Learnable weighted skip sum
        skip_w = F.softmax(self.log_skip_w, dim=0)           # (n_layers,)
        h_full = sum(w * s for w, s in zip(skip_w, skips))   # (B, 512, N, S)

        # ── Branch A: local info from last timestep ──────────────────────
        h_local = h_full[:, :, :, -1:]                        # (B, 512, N, 1)

        # ── Branch B: global temporal memory via GRU ─────────────────────
        B, d_sk, N, S = h_full.shape
        h_seq   = h_full.permute(0,2,3,1).reshape(B*N, S, d_sk)  # (B*N, S, 512)
        _, h_n  = self.gru(h_seq)                                  # h_n: (1, B*N, 256)
        h_gru   = h_n.squeeze(0).reshape(B, N, -1)                 # (B, N, 256)
        h_gru   = h_gru.permute(0, 2, 1).unsqueeze(-1)             # (B, 256, N, 1)

        # 5. Combine local (short-horizon) + GRU (long-horizon) features
        h_cat = torch.cat([h_local, h_gru], dim=1)                 # (B, 768, N, 1)

        # 6. Output MLP → (B, pred_len, N)
        out = F.relu(self.end_conv1(h_cat))                         # (B, 512, N, 1)
        out = self.end_conv2(out).squeeze(-1)                        # (B, pred_len, N)
        return out

print('GWGRUNet defined.')

In [ ]:
dl_train, dl_val, dl_test, mean_np, std_np, A_dist_np = build_dataloaders(cfg)

mean_flow = torch.from_numpy(mean_np[:, cfg.feature_idx]).to(device)  # (N,)
std_flow  = torch.from_numpy(std_np [:, cfg.feature_idx]).to(device)

set_seed()
model    = GWGRUNet(cfg, A_dist_np).to(device)
n_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f'\nGWGRU-Net v1 | {n_params:,} parameters')
print(f'd={cfg.d_model} | d_skip={cfg.d_skip} | d_gru={cfg.d_gru}')
print(f'n_layers={cfg.n_layers} | ks={cfg.kernel_size} | gcn_order={cfg.gcn_order}')
print(f'Baseline -> MAE=13.114 | RMSE=22.623 | MAPE=8.471%')

In [ ]:
# ═══════════════════════════════════════════════════════════════════════════
# METRICS — pool ALL predictions before computing (avoids batch-mean bias)
# ═══════════════════════════════════════════════════════════════════════════

def masked_huber(pred, true, delta=1.0):
    """Huber loss with zero-flow masking. Ignore timesteps where true=0."""
    mask = (true.abs() > 0.0).float()
    err  = torch.abs(pred - true)
    loss = torch.where(err < delta, 0.5*err**2, delta*(err - 0.5*delta))
    return (loss * mask).sum() / (mask.sum() + 1e-8)

def compute_mae(P, Y, thresh=0.0):
    mask = (Y.abs() > thresh).float()
    return (torch.abs(P-Y)*mask).sum() / (mask.sum()+1e-8)

def compute_rmse(P, Y, thresh=0.0):
    mask = (Y.abs() > thresh).float()
    return (((P-Y)**2*mask).sum() / (mask.sum()+1e-8)).sqrt()

def compute_mape(P, Y, thresh=10.0):
    mask = (Y.abs() > thresh).float()
    if mask.sum() < 1: return torch.tensor(0.0)
    return (torch.abs((P-Y)/(Y.abs()+1.0))*mask).sum() / mask.sum() * 100

print('Metrics defined (masked Huber, pooled-prediction).')

In [ ]:
scaler = torch.amp.GradScaler('cuda')

def train_epoch(model, loader, optimizer, device):
    model.train()
    total = 0.0
    for x_rec, x_hour, x_day, y, tod, dow in loader:
        x_rec  = x_rec .to(device, non_blocking=True)
        x_hour = x_hour.to(device, non_blocking=True)
        x_day  = x_day .to(device, non_blocking=True)
        y      = y     .to(device, non_blocking=True)
        tod    = tod   .to(device, non_blocking=True)
        dow    = dow   .to(device, non_blocking=True)
        optimizer.zero_grad(set_to_none=True)
        with torch.amp.autocast('cuda'):
            pred = model(x_rec, x_hour, x_day, tod, dow)
            loss = masked_huber(pred, y, delta=cfg.huber_delta)
        scaler.scale(loss).backward()
        scaler.unscale_(optimizer)
        torch.nn.utils.clip_grad_norm_(model.parameters(), cfg.grad_clip)
        scaler.step(optimizer); scaler.update()
        total += loss.item()
    return total / len(loader)


@torch.no_grad()
def eval_epoch(model, loader, device, mean_flow, std_flow):
    """Pool ALL batch predictions before computing metrics — avoids bias."""
    model.eval()
    all_pred, all_true = [], []
    for x_rec, x_hour, x_day, y, tod, dow in loader:
        x_rec  = x_rec .to(device, non_blocking=True)
        x_hour = x_hour.to(device, non_blocking=True)
        x_day  = x_day .to(device, non_blocking=True)
        tod    = tod   .to(device, non_blocking=True)
        dow    = dow   .to(device, non_blocking=True)
        with torch.amp.autocast('cuda'):
            pred = model(x_rec, x_hour, x_day, tod, dow)
        # Denormalise flow on CPU
        pd_ = pred.float().cpu() * std_flow.cpu()[None,None,:] + mean_flow.cpu()[None,None,:]
        yd_ = y   .float()       * std_flow.cpu()[None,None,:] + mean_flow.cpu()[None,None,:]
        all_pred.append(pd_); all_true.append(yd_)
    P = torch.cat(all_pred, dim=0)   # (total_samples, 12, 170)
    Y = torch.cat(all_true, dim=0)
    return (compute_mae(P,Y).item(), compute_rmse(P,Y).item(), compute_mape(P,Y).item())

print('Train / eval defined.')

In [ ]:
set_seed()

optimizer = torch.optim.AdamW(
    model.parameters(), lr=cfg.lr, weight_decay=cfg.weight_decay)

# ── LR schedule: Linear Warmup → ReduceLROnPlateau ────────────────────────
# WHY not cosine? Cosine keeps reducing LR on a fixed schedule even when
# val MAE is still improving — this is wasteful and can kill good runs.
# ReduceLROnPlateau only reduces when improvement stalls → more efficient.
def lr_lambda(ep):
    return min(1.0, (ep+1) / cfg.warmup_eps)

warmup_sched  = torch.optim.lr_scheduler.LambdaLR(optimizer, lr_lambda)
plateau_sched = torch.optim.lr_scheduler.ReduceLROnPlateau(
    optimizer, mode='min',
    patience=cfg.plat_patience, factor=cfg.plat_factor,
    min_lr=cfg.min_lr)

best_val_mae = best_val_rmse = best_val_mape = float('inf')
patience_cnt = 0
history = {'train_loss':[], 'val_mae':[], 'val_rmse':[], 'val_mape':[]}

print(f'GWGRU-Net v1 | {sum(p.numel() for p in model.parameters()):,} params')
print(f'd={cfg.d_model} | d_skip={cfg.d_skip} | d_gru={cfg.d_gru} | n_layers={cfg.n_layers}')
print(f'ks={cfg.kernel_size} | gcn_order={cfg.gcn_order} | adp_emb={cfg.adp_emb}')
print(f'3-stream additive | Weighted skip sum | GRU temporal decoder')
print(f'GroupNorm(16) | NO attention blocks (avoids overfitting)')
print(f'AdamW lr={cfg.lr} | warmup={cfg.warmup_eps}ep | ReduceLROnPlateau(p={cfg.plat_patience},f={cfg.plat_factor})')
print(f'Masked Huber delta={cfg.huber_delta} | grad_clip={cfg.grad_clip} | wd={cfg.weight_decay}')
print(f'epochs={cfg.epochs} | patience={cfg.patience}')
print(f'Baseline -> MAE=13.114 | RMSE=22.623 | MAPE=8.471%')
print('='*72)

for epoch in range(1, cfg.epochs+1):
    tr_loss = train_epoch(model, dl_train, optimizer, device)
    val_mae, val_rmse, val_mape = eval_epoch(model, dl_val, device, mean_flow, std_flow)

    # LR schedule
    if epoch <= cfg.warmup_eps:
        warmup_sched.step()
    else:
        plateau_sched.step(val_mae)

    history['train_loss'].append(tr_loss)
    history['val_mae'].append(val_mae)
    history['val_rmse'].append(val_rmse)
    history['val_mape'].append(val_mape)

    tag = ''
    if val_mae < best_val_mae:
        best_val_mae  = val_mae
        best_val_rmse = val_rmse
        best_val_mape = val_mape
        patience_cnt  = 0
        torch.save(model.state_dict(), cfg.best_path)
        tag = '  <- best'
    else:
        patience_cnt += 1
        if patience_cnt >= cfg.patience:
            print(f'Early stop at epoch {epoch}.')
            break

    lr_now = optimizer.param_groups[0]['lr']
    beat   = '  *** BEAT BASELINE! ***' if val_mae < 13.114 else ''
    if epoch % 5 == 0 or tag or beat:
        print(f'Ep {epoch:03d} | Loss={tr_loss:.4f} | '
              f'MAE={val_mae:.3f} RMSE={val_rmse:.3f} MAPE={val_mape:.2f}% '
              f'lr={lr_now:.1e}{tag}{beat}')

print(f'\nBest Val: MAE={best_val_mae:.3f} RMSE={best_val_rmse:.3f} MAPE={best_val_mape:.2f}%')
print(f'Baseline: MAE=13.114   RMSE=22.623   MAPE=8.471%')

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 4))
axes[0].plot(history['train_loss'], 'steelblue', label='Train Loss (Huber)')
axes[0].set_title('Training Loss'); axes[0].set_xlabel('Epoch'); axes[0].legend()
axes[1].plot(history['val_mae'], 'tab:orange', label='Val MAE')
axes[1].axhline(13.114, color='red', linestyle='--', label='Baseline 13.114')
axes[1].set_title('Validation MAE'); axes[1].set_xlabel('Epoch'); axes[1].legend()
axes[2].plot(history['val_rmse'], 'tab:green', label='Val RMSE')
axes[2].axhline(22.623, color='red', linestyle='--', label='Baseline 22.623')
axes[2].set_title('Validation RMSE'); axes[2].set_xlabel('Epoch'); axes[2].legend()
plt.tight_layout()
plt.savefig('gwgru_v1_curves.png', dpi=120); plt.show()
print('Saved: gwgru_v1_curves.png')

In [ ]:
model.load_state_dict(torch.load(cfg.best_path, map_location=device))

@torch.no_grad()
def paper_eval(model, loader, device, mean_flow, std_flow):
    """Pool all predictions → single metric computation (no batch-mean bias)."""
    model.eval()
    all_pred, all_true = [], []
    for x_rec, x_hour, x_day, y, tod, dow in loader:
        x_rec  = x_rec.to(device); x_hour = x_hour.to(device)
        x_day  = x_day.to(device); tod = tod.to(device); dow = dow.to(device)
        pred   = model(x_rec, x_hour, x_day, tod, dow)
        pd_ = pred.float().cpu() * std_flow.cpu()[None,None,:] + mean_flow.cpu()[None,None,:]
        yd_ = y   .float()       * std_flow.cpu()[None,None,:] + mean_flow.cpu()[None,None,:]
        all_pred.append(pd_); all_true.append(yd_)
    P = torch.cat(all_pred); Y = torch.cat(all_true)
    mae  = compute_mae(P, Y).item()
    rmse = compute_rmse(P, Y).item()
    mape = compute_mape(P, Y).item()
    print('\n' + '='*60)
    print('  TEST RESULTS  (averaged over all 12 horizons)')
    print('='*60)
    print(f'  MAE  : {mae:.3f}    baseline: 13.114   Delta={mae-13.114:+.3f}')
    print(f'  RMSE : {rmse:.3f}    baseline: 22.623   Delta={rmse-22.623:+.3f}')
    print(f'  MAPE : {mape:.3f}%   baseline:  8.471%  Delta={mape-8.471:+.3f}%')
    print('='*60)
    all_beat = mae < 13.114 and rmse < 22.623 and mape < 8.471
    if all_beat: print('  *** BEATS MD-GRTN ON ALL THREE METRICS! ***')
    elif mae < 13.114: print('  *** BEATS BASELINE MAE! ***')
    return mae, rmse, mape

paper_eval(model, dl_test, device, mean_flow, std_flow)

In [ ]:
@torch.no_grad()
def horizon_eval(model, loader, device, mean_flow, std_flow):
    model.eval()
    all_pred, all_true = [], []
    for x_rec, x_hour, x_day, y, tod, dow in loader:
        x_rec = x_rec.to(device); x_hour = x_hour.to(device)
        x_day = x_day.to(device); tod = tod.to(device); dow = dow.to(device)
        pred = model(x_rec, x_hour, x_day, tod, dow)
        pd_ = pred.float().cpu() * std_flow.cpu()[None,None,:] + mean_flow.cpu()[None,None,:]
        yd_ = y   .float()       * std_flow.cpu()[None,None,:] + mean_flow.cpu()[None,None,:]
        all_pred.append(pd_); all_true.append(yd_)
    P = torch.cat(all_pred); Y = torch.cat(all_true)  # (N_s, 12, 170)
    print(f'\nHorizon breakdown (5-min steps):')
    print(f'{"Step":>6} | {"MAE":>7} | {"RMSE":>7} | {"MAPE":>7}')
    print('-'*36)
    for t in range(P.shape[1]):
        mae  = compute_mae (P[:,t,:], Y[:,t,:]).item()
        rmse = compute_rmse(P[:,t,:], Y[:,t,:]).item()
        mape = compute_mape(P[:,t,:], Y[:,t,:]).item()
        print(f'{(t+1)*5:>4}min | {mae:>7.3f} | {rmse:>7.3f} | {mape:>6.2f}%')

horizon_eval(model, dl_test, device, mean_flow, std_flow)